#### **Developer**: Pedro Augusto Andrade Silva
#### **Objectives**: Outlook automation of e-mail sending with different attachments and recipient list, considering the business days and with a customizable e-mail body.

## 1) Libraries used

In [6]:
import calendar
import locale
import time
from datetime import datetime, timedelta, date
import win32com.client as win32
import pandas as pd
import holidays

## 2) Initial settings

In [7]:
locale.setlocale(locale.LC_ALL, "pt_BR.utf8")
mes_atual_nome = calendar.month_name[datetime.today().month].title()


br_holidays = holidays.Brazil(years = datetime.today().year)
feriados = [date for date, _ in sorted(br_holidays.items())]

nome_arquivos = [
    'anexo1',
    'anexo2',
    'anexo3',
]

anexos = [
    r"C:\Users\pedro\OneDrive\Área de Trabalho\Pedro\Projetos\automacoes\outlook\anexo1.xlsx",
    r"C:\Users\pedro\OneDrive\Área de Trabalho\Pedro\Projetos\automacoes\outlook\anexo2.xlsx",
    r"C:\Users\pedro\OneDrive\Área de Trabalho\Pedro\Projetos\automacoes\outlook\anexo3.xlsx",
]


## 3) Functions

In [8]:
def calcular_data_entrega(data_atual, feriados):
    dias_uteis = 0
    while dias_uteis < 2:
        data_entrega = data_atual + timedelta(days=1)
        if data_entrega.weekday() < 5 and data_entrega not in feriados:
            dias_uteis += 1
        data_atual = data_entrega
    return data_entrega

def exibir_mensagem_sucesso(arquivo):
    time.sleep(1)
    print(f"O arquivo \"{arquivo}\" foi enviado com sucesso!")

data_atual = datetime.now().date()
data_entrega = calcular_data_entrega(data_atual, feriados)
data_entrega_formatada = data_entrega.strftime("%d-%m")

## 4) Execution

In [9]:
outlook = win32.Dispatch('outlook.application')

with open("corpo_email.txt", "r", encoding="utf-8") as arquivo_txt:
    texto_personalizado = arquivo_txt.read()

for i in range(3):
    df_emails_to = pd.read_excel(r"C:\Users\pedro\OneDrive\Área de Trabalho\Pedro\Projetos\automacoes\outlook\destinatarios.xlsx", sheet_name=0, header=0)
    df_emails_cc = pd.read_excel(r"C:\Users\pedro\OneDrive\Área de Trabalho\Pedro\Projetos\automacoes\outlook\destinatarios.xlsx", sheet_name=1, header=0)
    df_emails_bcc = pd.read_excel(r"C:\Users\pedro\OneDrive\Área de Trabalho\Pedro\Projetos\automacoes\outlook\destinatarios.xlsx", sheet_name=2, header=0)
    destinatarios_to = list(df_emails_to[f"Grupo {i+1}"])
    destinatarios_cc = list(df_emails_cc[f"Grupo {i+1}"])
    destinatarios_bcc = list(df_emails_bcc[f"Grupo {i+1}"])

    email = outlook.CreateItem(0)
    email.To = ';'.join(destinatarios_to)
    email.CC = ';'.join(destinatarios_cc)
    email.BCC = ';'.join(destinatarios_bcc)
    email.Subject = f"Envio e-mails - {mes_atual_nome}/23"
    email.Body = f"""Prezados,

Gentileza nos enviar o documento em um prazo de até 2 dias úteis, ou seja, até o dia {data_entrega_formatada}.

{texto_personalizado}"""

    email.Attachments.Add(anexos[i])
    email.Send()
    
    exibir_mensagem_sucesso(nome_arquivos[i])

O arquivo "anexo1" foi enviado com sucesso!
O arquivo "anexo2" foi enviado com sucesso!
O arquivo "anexo3" foi enviado com sucesso!
